In [ ]:
# !which python
!python --version
!pip freeze > requirements.txt

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src" / "data" / "SmartEM" / "Philips").exists():
    repo_root = repo_root.parents[2]

data_dir = repo_root / "src" / "data" / "SmartEM" / "Philips"

input_data = pd.read_csv(data_dir / "input.csv")
output_data = pd.read_csv(data_dir / "output.csv")
grid_data = pd.read_csv(data_dir / "PATH_Ref_62_CatchRing.csv")
# input_data, output_data

# Visualisation

In [ ]:
input_data.nunique()
# input_data.info()
input_data.head()
# input_data = input_data.loc[:, input_data.nunique() > 1]
# input_21 is degrees and should be converted to radians
# input_data.iloc[:, 21] = np.radians(input_data.iloc[:, 21])
# input_data.describe()
#input_20 is initial hair length
# input_14 is elevation angle is in degrees
# High 1 10
# Second High 11 3 9 2 4 6
# Third High 18 19 
input_data.nunique()
# Drop columns with only one unique value
# input_data = input_data.loc[:, input_data.nunique() > 1]
input_data.nunique()
# input_data.head()
# Number of rows in input_data and output_data
# len(input_data), len(output_data)
grid_data.describe()
# grid_data.nunique()
output_data.describe()
# input_data.nunique()
# output_data.nunique() 

In [ ]:

import numpy as np
x_col = "x_begin1_hair"
y_col = "y_begin1_hair"
dx_col = "dx1_hair"
dy_col = "dy1_hair"

x_valsbegin = grid_data["x_begin1_hair"].to_numpy()
y_valsbegin = grid_data["y_begin1_hair"].to_numpy()

x_valsend = x_valsbegin - grid_data["dx1_hair"].to_numpy()
y_valsend = y_valsbegin - grid_data["dy1_hair"].to_numpy()

x_vals = np.sort(np.unique(np.concatenate([x_valsbegin, x_valsend])))
y_vals = np.sort(np.unique(np.concatenate([y_valsbegin, y_valsend])))

X, Y = np.meshgrid(x_vals, y_vals)
XYPos = np.stack((X, Y), axis=-1)

ny = len(y_vals)
nx = len(x_vals)

XYGridBegin = np.zeros((ny, nx), dtype=int)
XYGridEnd = np.zeros((ny, nx), dtype=int)
XYGridDXY = np.zeros((ny, nx, 2), dtype=float)

n = len(grid_data)
inputHairs = np.zeros((ny, nx), dtype=int)
for k in range(n):
    x = grid_data.loc[k, x_col]
    y = grid_data.loc[k, y_col]
    dx = grid_data.loc[k, dx_col]
    dy = grid_data.loc[k, dy_col]

    row = np.where(np.isclose(y_vals, y))[0][0]
    col = np.where(np.isclose(x_vals, x))[0][0]

    XYGridBegin[row, col] = 1
    XYGridDXY[row, col] = (dx, dy)

    x_end = x - dx
    y_end = y - dy

    row = np.where(np.isclose(y_vals, y_end))[0][0]
    col = np.where(np.isclose(x_vals, x_end))[0][0]

    XYGridEnd[row, col] = 1

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x_min = grid_data[x_col].min()
y_min = grid_data[y_col].min()

x_max = (grid_data[x_col]).max()
y_max = (grid_data[y_col]).max()

common_extent = [x_min, x_max, y_min, y_max]

plt.figure()
plt.imshow(XYGridBegin, origin="lower", extent=common_extent, aspect="auto")
plt.colorbar()
plt.xlabel("x")
plt.ylabel("y")
plt.title("XYGridBegin")
plt.show()

imgXYGridDXY = np.linalg.norm(XYGridDXY, axis=-1)

plt.figure()
plt.imshow(imgXYGridDXY, origin="lower", extent=common_extent, aspect="auto")
plt.colorbar()
plt.xlabel("x")
plt.ylabel("y")
plt.title("XYGridDXY magnitude")
plt.show()

plt.figure()
plt.imshow(XYGridEnd, origin="lower", extent=common_extent, aspect="auto")
plt.colorbar()
plt.xlabel("x")
plt.ylabel("y")
plt.title("XYGridEnd")
plt.show()

# Placing input_data in Grids
1) The output will be a collection of X "images" with 20 channels, grid size 62 by 31. 
2) X will be how many "training inputs" that are there

In [ ]:

import numpy as np
# Input Data

x_col = "input_18"
y_col = "input_19"
channel_cols = [c for c in input_data.columns if c not in [x_col, y_col]]

x_vals = np.sort(input_data[x_col].unique())
y_vals = np.sort(input_data[y_col].unique())

nx = len(x_vals)
ny = len(y_vals)
n_channels = len(channel_cols)

points_per_image = nx * ny
n_images = len(input_data) // points_per_image

images = np.empty((n_images, ny, nx, n_channels), dtype=object)
images[:] = None

x_to_col = {x: i for i, x in enumerate(x_vals)}
y_to_row = {y: i for i, y in enumerate(y_vals)}

for img_i in range(n_images):
    df_img = input_data.iloc[
        img_i * points_per_image : (img_i + 1) * points_per_image
    ]

    for _, row in df_img.iterrows():
        r = y_to_row[row[y_col]]
        c = x_to_col[row[x_col]]

        images[img_i, r, c, :] = row[channel_cols].to_numpy(dtype=object)

print(images.shape)
print(channel_cols)

print("Created Input Data. Making Output Data next")
# Output Data
output_channel_cols = output_data.columns[:3].tolist()

outputs = np.full((n_images, ny, nx, 3), np.nan, dtype=float)

for img_i in range(n_images):
    df_in_img = input_data.iloc[
        img_i * points_per_image : (img_i + 1) * points_per_image
    ]

    df_out_img = output_data.iloc[
        img_i * points_per_image : (img_i + 1) * points_per_image
    ]

    for local_i, (_, row) in enumerate(df_in_img.iterrows()):
        r = y_to_row[row[y_col]]
        c = x_to_col[row[x_col]]

        outputs[img_i, r, c, 0] = int(df_out_img.iloc[local_i]["output_1"])    # binary
        outputs[img_i, r, c, 1] = int(df_out_img.iloc[local_i]["output_2"])    # integer
        outputs[img_i, r, c, 2] = float(df_out_img.iloc[local_i]["output_3"])  # float

print(outputs.shape)
print(output_channel_cols)

# Visualising the Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def channel_to_image(A, colname):
    flat = A.ravel()

    categorical_inputs = [1, 7, 8, 9, 10, 12]
    integer_inputs = [2, 3, 4, 5, 6, 11, 13, 15, 16, 17]

    categorical_cols = [f"input_{i}" for i in categorical_inputs]
    integer_cols = [f"input_{i}" for i in integer_inputs]

    if colname in categorical_cols:
        codes, uniques = pd.factorize(flat)
        return codes.reshape(A.shape), "categorical"

    elif colname in integer_cols:
        numeric = pd.to_numeric(flat, errors="coerce")
        return np.asarray(numeric).reshape(A.shape), "integer"

    else:
        numeric = pd.to_numeric(flat, errors="coerce")
        return np.asarray(numeric).reshape(A.shape), "float"


def plot_input_output_sample(images, outputs, sample_idx, channel_cols, output_channel_cols):
    x_min = input_data[x_col].min()
    y_min = input_data[y_col].min()
    x_max = input_data[x_col].max()
    y_max = input_data[y_col].max()

    common_extent = [x_min, x_max, y_min, y_max]

    fig, axes = plt.subplots(5, 5, figsize=(16, 15))
    axes = axes.ravel()

    # 20 input channels
    for ch in range(20):
        colname = channel_cols[ch]
        A = images[sample_idx, :, :, ch]

        img, kind = channel_to_image(A, colname)

        ax = axes[ch]
        im = ax.imshow(
            img,
            origin="lower",
            aspect="auto",
            extent=common_extent
        )

        ax.set_title(f"IN {colname} ({kind})")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # 3 output channels
    output_vmins = np.nanmin(outputs, axis=(0, 1, 2))
    output_vmaxs = np.nanmax(outputs, axis=(0, 1, 2))
    for ch in range(3):
        colname = output_channel_cols[ch]
        A = outputs[sample_idx, :, :, ch]

        ax = axes[20 + ch]
        im = ax.imshow(
            A,
            origin="lower",
            aspect="auto",
            extent=common_extent,
            vmin=output_vmins[ch],
            vmax=output_vmaxs[ch]
        )

        if ch == 0:
            kind = "binary"
        elif ch == 1:
            kind = "integer"
        else:
            kind = "float"

        ax.set_title(f"OUT {colname} ({kind})")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # hide empty plots
    for ax in axes[23:]:
        ax.axis("off")

    fig.suptitle(f"Input + Output Sample {sample_idx}", fontsize=16)
    plt.tight_layout()

    return fig
plot_input_output_sample(images, outputs, 0, channel_cols, output_channel_cols)

# Creating PDF file of Data

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
num_imgs = images.shape[0]

with PdfPages("data.pdf") as pdf:
    for i in range(num_imgs):
        fig = plot_input_output_sample(
            images,
            outputs,
            i,
            channel_cols,
            output_channel_cols
        )

        pdf.savefig(fig)
        plt.close(fig)
        print(f"Saved Image {i+1} of {num_imgs}")



# Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def preprocess_input(
    input_data,
    encodingHigh=0,
    categoricalEncodingHighType="one-hot",
    encodingMed=0,
    categoricalEncodingMedType="one-hot",
    encodingLow=0,
    categoricalEncodingLowType="one-hot",
    convertToRadians=0
):
    df = input_data.copy()

    # If nothing is enabled, return unchanged
    if (
        encodingHigh == 0 and
        encodingMed == 0 and
        encodingLow == 0 and
        convertToRadians == 0
    ):
        return df

    # Helper function
    def encode_columns(df, cols, encoding_type):
        cols = [col for col in cols if col in df.columns]

        if len(cols) == 0:
            return df

        if encoding_type is None:
            encoding_type = "one-hot"

        encoding_type = encoding_type.lower()

        if encoding_type == "one-hot":
            df = pd.get_dummies(df, columns=cols)

        elif encoding_type == "label":
            le = LabelEncoder()
            for col in cols:
                df[col] = le.fit_transform(df[col].astype(str))

        return df

    # High group
    if encodingHigh != 0:
        high_cols = ["input_10"]
        df = encode_columns(df, high_cols, categoricalEncodingHighType)

    # Medium group
    if encodingMed != 0:
        med_cols = [
            "input_2", "input_3", "input_4",
            "input_6", "input_8","input_9", "input_11",
            "input_10"
        ]
        df = encode_columns(df, med_cols, categoricalEncodingMedType)

    # Low group
    if encodingLow != 0:
        low_cols = ["input_18", "input_19"]
        df = encode_columns(df, low_cols, categoricalEncodingLowType)

    # Convert angles to radians
    if convertToRadians != 0:
        for col in ["input_14", "input_21"]:
            if col in df.columns:
                df[col] = np.deg2rad(df[col])
    return df

# Train Test Split

In [ ]:
import numpy as np

def trainTestSplit(processedInput, output, Train=10, Test=3, seed=42):
    # Unique values of input_1
    unique_vals = processedInput["input_1"].unique()

    # Check count
    if len(unique_vals) != Train + Test:
        raise ValueError(
            f"Found {len(unique_vals)} unique values in input_1, "
            f"but Train + Test = {Train + Test}"
        )

    # Shuffle with seed
    rng = np.random.RandomState(seed)
    shuffled_vals = unique_vals.copy()
    rng.shuffle(shuffled_vals)

    # Split
    TrainVals = shuffled_vals[:Train]
    TestVals = shuffled_vals[Train:]

    # Create train and test sets
    input_train = processedInput[processedInput["input_1"].isin(TrainVals)].copy()
    input_test = processedInput[processedInput["input_1"].isin(TestVals)].copy()
    output_train = output[processedInput["input_1"].isin(TrainVals)].copy()
    output_test = output[processedInput["input_1"].isin(TestVals)].copy()

    # Drop input_1 after splitting
    input_train = input_train.drop(columns=["input_1"])
    input_test = input_test.drop(columns=["input_1"])

    return input_train, input_test,output_train,output_test, TrainVals, TestVals


input_data.nunique()
# Drop columns with only one unique value
input_data = input_data.loc[:, input_data.nunique() > 1]
input_data.nunique()

processed_input = preprocess_input(input_data, encodingHigh=1, categoricalEncodingHighType="one-hot", encodingMed=1, categoricalEncodingMedType="one-hot", encodingLow=1, categoricalEncodingLowType="one-hot", convertToRadians=1)
input_train, input_test,output_train,output_test, TrainVals, TestVals = trainTestSplit(processedInput=processed_input, output=output_data, Train=10, Test=3, seed=42)
input_train.head()


# TabPFN
Does not work. Licensed tool. Not paying for this.

In [ ]:
# from tabpfn import TabPFNClassifier, TabPFNRegressor

# # clf = TabPFNClassifier()
# # clf.fit(input_train, output_data)  # downloads checkpoint on first use
# # predictions = clf.predict(input_test)

# reg = TabPFNRegressor()
# reg.fit(input_train, output_data)  # downloads checkpoint on first use
# predictions = reg.predict(input_test)

# Tree Models Initialisation

In [ ]:
from sklearn.model_selection import train_test_split # Function to split data into training and testing sets
import numpy as np


from xgboost import XGBRegressor

def train_xgboost(X_train, y_train):
    model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    enable_categorical=True,
    verbosity=2
)

    model.fit(X_train, y_train, verbose=True)
    return model

from lightgbm import LGBMRegressor
def train_lightgbm(X_train, y_train):
    model = LGBMRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)
    return model

from catboost import CatBoostRegressor
def train_catboost(X_train, y_train, categorical_columns=None):
    model = CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=10
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_columns
    )

    return model

In [ ]:
output_data.head()

# Training on Output_1

In [ ]:
# X_train = Xtrain[0]
# y_train = Ytrain[0]["output_1"]
# reg.fit(input_train, output_data)  # downloads checkpoint on first use
# input_train, input_test,output_train,output_test, TrainVals, TestVals 
xgb_model = train_xgboost(input_train, output_train["output_1"])
lgb_model = train_lightgbm(input_train, output_train["output_1"])
cat_model = train_catboost(input_train, output_train["output_1"])

# Testing

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Predictions
xgb_pred = xgb_model.predict(input_test)
lgb_pred = lgb_model.predict(input_test)
cat_pred = cat_model.predict(input_test)

y_true = output_test["output_1"].values

# Metrics
models = {
    "XGBoost": xgb_pred,
    "LightGBM": lgb_pred,
    "CatBoost": cat_pred
}

r2_scores = []
mae_scores = []
rmse_scores = []

for name, pred in models.items():
    r2_scores.append(r2_score(y_true, pred))
    mae_scores.append(mean_absolute_error(y_true, pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_true, pred)))

# Plot metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(models.keys(), r2_scores)
axes[0].set_title("R² Score")
axes[0].set_ylabel("Higher is better")

axes[1].bar(models.keys(), mae_scores)
axes[1].set_title("MAE")
axes[1].set_ylabel("Lower is better")

axes[2].bar(models.keys(), rmse_scores)
axes[2].set_title("RMSE")
axes[2].set_ylabel("Lower is better")

plt.tight_layout()
plt.show()


# Scatter plot: true vs predicted for each model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, models.items()):
    ax.scatter(y_true, pred, alpha=0.7)

    # ideal line
    min_val = min(y_true.min(), pred.min())
    max_val = max(y_true.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--')

    ax.set_title(name)
    ax.set_xlabel("True output_1")
    ax.set_ylabel("Predicted output_1")

plt.tight_layout()
plt.show()

# Training Output 2

In [ ]:
# Training on Output_1
# X_train = Xtrain[0]
# y_train = Ytrain[0]["output_1"]
# reg.fit(input_train, output_data)  # downloads checkpoint on first use
# input_train, input_test,output_train,output_test, TrainVals, TestVals 
xgb_model = train_xgboost(input_train, output_train["output_2"])
lgb_model = train_lightgbm(input_train, output_train["output_2"])
cat_model = train_catboost(input_train, output_train["output_2"])

# Testing Output 2

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Predictions
xgb_pred = xgb_model.predict(input_test)
lgb_pred = lgb_model.predict(input_test)
cat_pred = cat_model.predict(input_test)

y_true = output_test["output_2"].values

# Metrics
models = {
    "XGBoost": xgb_pred,
    "LightGBM": lgb_pred,
    "CatBoost": cat_pred
}

r2_scores = []
mae_scores = []
rmse_scores = []

for name, pred in models.items():
    r2_scores.append(r2_score(y_true, pred))
    mae_scores.append(mean_absolute_error(y_true, pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_true, pred)))

# Plot metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(models.keys(), r2_scores)
axes[0].set_title("R² Score")
axes[0].set_ylabel("Higher is better")

axes[1].bar(models.keys(), mae_scores)
axes[1].set_title("MAE")
axes[1].set_ylabel("Lower is better")

axes[2].bar(models.keys(), rmse_scores)
axes[2].set_title("RMSE")
axes[2].set_ylabel("Lower is better")

plt.tight_layout()
plt.show()


# Scatter plot: true vs predicted for each model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, models.items()):
    ax.scatter(y_true, pred, alpha=0.7)

    # ideal line
    min_val = min(y_true.min(), pred.min())
    max_val = max(y_true.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--')

    ax.set_title(name)
    ax.set_xlabel("True output_2")
    ax.set_ylabel("Predicted output_2")

plt.tight_layout()
plt.show()

# Training Output 3

In [ ]:
# Training on Output_1
# X_train = Xtrain[0]
# y_train = Ytrain[0]["output_1"]
# reg.fit(input_train, output_data)  # downloads checkpoint on first use
# input_train, input_test,output_train,output_test, TrainVals, TestVals 
xgb_model = train_xgboost(input_train, output_train["output_3"])
lgb_model = train_lightgbm(input_train, output_train["output_3"])
cat_model = train_catboost(input_train, output_train["output_3"])

# Testing Output 3

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Predictions
xgb_pred = xgb_model.predict(input_test)
lgb_pred = lgb_model.predict(input_test)
cat_pred = cat_model.predict(input_test)

y_true = output_test["output_3"].values

# Metrics
models = {
    "XGBoost": xgb_pred,
    "LightGBM": lgb_pred,
    "CatBoost": cat_pred
}

r2_scores = []
mae_scores = []
rmse_scores = []

for name, pred in models.items():
    r2_scores.append(r2_score(y_true, pred))
    mae_scores.append(mean_absolute_error(y_true, pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_true, pred)))

# Plot metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].bar(models.keys(), r2_scores)
axes[0].set_title("R² Score")
axes[0].set_ylabel("Higher is better")

axes[1].bar(models.keys(), mae_scores)
axes[1].set_title("MAE")
axes[1].set_ylabel("Lower is better")

axes[2].bar(models.keys(), rmse_scores)
axes[2].set_title("RMSE")
axes[2].set_ylabel("Lower is better")

plt.tight_layout()
plt.show()


# Scatter plot: true vs predicted for each model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, models.items()):
    ax.scatter(y_true, pred, alpha=0.7)

    # ideal line
    min_val = min(y_true.min(), pred.min())
    max_val = max(y_true.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--')

    ax.set_title(name)
    ax.set_xlabel("True output_3")
    ax.set_ylabel("Predicted output_3")

plt.tight_layout()
plt.show()

# Auto Gluon

In [ ]:
!pip install -U pip
!pip install -U setuptools wheel
!pip install autogluon

# Import Data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src" / "data" / "SmartEM" / "Philips").exists():
    repo_root = repo_root.parents[2]

data_dir = repo_root / "src" / "data" / "SmartEM" / "Philips"

input_data = pd.read_csv(data_dir / "input.csv")
output_data = pd.read_csv(data_dir / "output.csv")

# Train Test Split
Apparently AutoGluon does not need pre processing

In [ ]:
import numpy as np

def trainTestSplit(processedInput, output, Train=10, Test=3, seed=42):
    # Unique values of input_1
    unique_vals = processedInput["input_1"].unique()

    # Check count
    if len(unique_vals) != Train + Test:
        raise ValueError(
            f"Found {len(unique_vals)} unique values in input_1, "
            f"but Train + Test = {Train + Test}"
        )

    # Shuffle with seed
    rng = np.random.RandomState(seed)
    shuffled_vals = unique_vals.copy()
    rng.shuffle(shuffled_vals)

    # Split
    TrainVals = shuffled_vals[:Train]
    TestVals = shuffled_vals[Train:]

    # Create train and test sets
    input_train = processedInput[processedInput["input_1"].isin(TrainVals)].copy()
    input_test = processedInput[processedInput["input_1"].isin(TestVals)].copy()
    output_train = output[processedInput["input_1"].isin(TrainVals)].copy()
    output_test = output[processedInput["input_1"].isin(TestVals)].copy()

    # Drop input_1 after splitting
    input_train = input_train.drop(columns=["input_1"])
    input_test = input_test.drop(columns=["input_1"])

    return input_train, input_test,output_train,output_test, TrainVals, TestVals



input_train, input_test,output_train,output_test, TrainVals, TestVals = trainTestSplit(processedInput=input_data, output=output_data, Train=10, Test=3, seed=42)
input_train.head()


# Dataset Wrapper
Not required, but it is a way to standardise inputs to autogluon

In [ ]:
from autogluon.tabular import TabularDataset
input_train_data = TabularDataset(input_train)
output_train_data = TabularDataset(output_train)
input_test_data = TabularDataset(input_test)
output_test_data = TabularDataset(output_test)

# Train

In [ ]:
from autogluon.tabular import TabularPredictor

predictors = {}
ag_args_fit = {"num_gpus": 1}

hyperparameters = {
    "CAT": {
        "task_type": "GPU",
        "devices": "0",
        "ag_args_fit": ag_args_fit
    }
}
itr=0
# Need to train one model per output
for col in output_train.columns:
    # combine input + one output
    train_data = input_train.copy()
    train_data[col] = output_train[col]
    train_data = TabularDataset(train_data)
    print(f"Training predictor for {col}...")
    # train one predictor per output
    # predictors[col] = TabularPredictor(label=col).fit(train_data)
    probType=["binary", "regression", "regression"]
    predictor = TabularPredictor(label=col).fit(
        train_data,
        presets="extreme",
        num_gpus=1,
        hyperparameters=hyperparameters,
        num_bag_folds=0,
        problem_type=probType[itr]
    )
    itr += 1

# Test

In [ ]:
predictor3 = TabularPredictor.load("/home/sa1/Documents/gitReps/pythonQuickAndDirty/src/scripts/SmartEM/AutogluonModels/ag-20260423_124824")
predictor2 = TabularPredictor.load("/home/sa1/Documents/gitReps/pythonQuickAndDirty/src/scripts/SmartEM/AutogluonModels/ag-20260423_124650")
predictor1 = TabularPredictor.load("/home/sa1/Documents/gitReps/pythonQuickAndDirty/src/scripts/SmartEM/AutogluonModels/ag-20260423_124540")

# predictor1.info()
predictors = {
    "output_1": predictor1,
    "output_2": predictor2,
    "output_3": predictor3
}

for col, predictor in predictors.items():
    y_pred = predictor.predict(input_test)

# Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for col in ["output_2", "output_3"]:
    y_true = output_test[col].values
    y_pred = predictors[col].predict(input_test)

    plt.figure()
    plt.scatter(y_true, y_pred)
    
    # ideal line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val])
    
    plt.xlabel("True")
    plt.ylabel("Predicted")
    plt.title(f"Regression Plot: {col}")
    plt.show()


from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = output_test["output_1"]
y_pred = predictors["output_1"].predict(input_test)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm)

disp.plot()
plt.title("Confusion Matrix: output_1")
plt.show()

from sklearn.metrics import roc_curve, auc

y_true = output_test["output_1"]
y_proba = predictors["output_1"].predict_proba(input_test)[1]  # prob of class 1

fpr, tpr, _ = roc_curve(y_true, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1])  # random baseline

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: output_1")
plt.legend()
plt.show()



residuals = y_true - y_pred

plt.figure()
plt.scatter(y_pred, residuals)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title(f"Residual Plot: {col}")
plt.show()

# Image generation
1) Create coordinate grid
   1) All unique pairs (input_18, input_19) forms the backbone grid
2) Create unique classes
   1) All unique elements in input_1 is a class
3) For each unique class, construct an image with (22-3)input  + 3 output channels with the grid coordinate templates.
   1) The -3 is because the inputs 18,19 and 1 are no longer to be considered
4) For every grid coordinate missing, leave the image and channel as NaN
5) For every duplicate coordinate, construct a new image with (22-3) inputs + 3 outputs channels.

```
Tabular rows
    ↓
Group rows by input_1
    ↓
Map each row to a pixel using input_18 and input_19
    ↓
Store remaining input variables as input image channels
    ↓
Store output_1, output_2, output_3 as output image channels
```

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src" / "data" / "SmartEM" / "Philips").exists():
    repo_root = repo_root.parents[2]

data_dir = repo_root / "src" / "data" / "SmartEM" / "Philips"

input_data = pd.read_csv(data_dir / "input.csv")
output_data = pd.read_csv(data_dir / "output.csv")
grid_data = pd.read_csv(data_dir / "PATH_Ref_62_CatchRing.csv")
# input_data, output_data

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Image generation
# -----------------------------

class_col = "input_1"
x_col = "input_18"
y_col = "input_19"

output_channel_cols = ["output_1", "output_2", "output_3"]
# All input columns except class_col, x_col, y_col becomes a "channel"
input_channel_cols = [
    c for c in input_data.columns
    if c not in [class_col, x_col, y_col]
]

# Collects every unique coordinate pair appearing in the entire dataset.
coord_pairs = (
    input_data[[x_col, y_col]]
    .drop_duplicates()
    .sort_values([y_col, x_col])
    .reset_index(drop=True)
)

x_vals = np.sort(coord_pairs[x_col].unique())
y_vals = np.sort(coord_pairs[y_col].unique())

height = len(y_vals)
width = len(x_vals)

# These dictionaries translate physical coordinates into image indices.
x_to_col = {x: i for i, x in enumerate(x_vals)}
y_to_row = {y: i for i, y in enumerate(y_vals)}

# Marks which rectangular grid cells are real coordinate pairs
valid_coord_mask = np.zeros((height, width), dtype=bool)

for _, row in coord_pairs.iterrows():
    r = y_to_row[row[y_col]]
    c = x_to_col[row[x_col]]
    valid_coord_mask[r, c] = True

input_blocks = []
output_blocks = []
image_input_1 = []
# Check physical validity before generating any images:
# output_3 is cut_length and input_20 is initial_length.
# A cut hair cannot become longer than its initial length.
# Equality and floating-point differences up to 1e-9 are allowed.
length_tolerance = 1e-9

length_difference = output_data["output_3"] - input_data["input_20"]

invalid_length_mask = length_difference > length_tolerance

if invalid_length_mask.any():
    invalid_rows = pd.DataFrame({
        class_col: input_data.loc[invalid_length_mask, class_col],
        x_col: input_data.loc[invalid_length_mask, x_col],
        y_col: input_data.loc[invalid_length_mask, y_col],
        "input_20_initial_length": input_data.loc[invalid_length_mask, "input_20"],
        "output_3_cut_length": output_data.loc[invalid_length_mask, "output_3"],
        "output_minus_input": length_difference.loc[invalid_length_mask],
    })

    raise ValueError(
        "Invalid hair-length data found: output_3 (cut_length) exceeds "
        "input_20 (initial_length) by more than 1e-9 for at least one row. "
        "A hair cannot be longer after cutting.\n\n"
        f"{invalid_rows.to_string(index=True)}"
    )
# processes rows separately for each value of input_1.
for class_value, df_class in input_data.groupby(class_col, sort=False, dropna=False):
    df_class = df_class.copy()

    # Copy the corresponding output_3 values into df_class so they can be used for sorting.
    df_class["_sort_output_3"] = output_data.loc[df_class.index, "output_3"]

    # Within each coordinate pair, rank rows from largest output hair length
    # to smallest output hair length.
    # input_20 is used only if two rows have the same output_3.
    df_class = df_class.sort_values(
        by=[x_col, y_col, "_sort_output_3", "input_20"],
        ascending=[True, True, False, False],
        kind="mergesort",
    )

    # Duplicate coordinate PAIRS within this input_1 class become extra images.
    # Within one input_1 group, the same coordinate pair may appear more than once.
    # After sorting, _duplicate_i is now the sequence rank within each independent pixel:
    # 0 = highest output_3, 1 = second highest output_3, ..., last = lowest input_20.
    df_class["_duplicate_i"] = df_class.groupby(
        [x_col, y_col],
        sort=False,
        dropna=False,
    ).cumcount()

    # repeated rows become additional images.
    # Each image now represents one rank layer across all independent pixels:
    # image 0 = largest output_3 at each pixel,
    # image 1 = second-largest output_3 at each pixel, and so on.
    n_class_images = int(df_class["_duplicate_i"].max()) + 1

    # For each input_1 group, the input image tensor has shape:
    # (number_of_images, height, width, number_of_input_channels)
    print(
        f"input_1 = {class_value}: "
        f"{n_class_images} ranked images generated, "
        f"from longest to shortest per coordinate"
    )

    class_input_images = np.full(
        (n_class_images, height, width, len(input_channel_cols)),
        np.nan,
        dtype=object,
    )

    # (number_of_images, height, width, 3)
    class_output_images = np.full(
        (n_class_images, height, width, len(output_channel_cols)),
        np.nan,
        dtype=float,
    )

    # Each row is assigned to the appropriate image position.
    # Rows with missing coordinates are skipped.
    for idx, row in df_class.iterrows():
        x = row[x_col]
        y = row[y_col]

        if pd.isna(x) or pd.isna(y):
            continue

        # img_i is the rank of this observation within its coordinate pair.
        # 0 = largest output_3, 1 = second-largest output_3, etc.
        img_i = int(row["_duplicate_i"])

        r = y_to_row[y]
        c = x_to_col[x]

        # All input variables for that row are placed into the corresponding pixel location.
        # pixel at position (r, c) = [input channels]
        class_input_images[img_i, r, c, :] = row[input_channel_cols].to_numpy(dtype=object)

        out_row = output_data.loc[idx]
        class_output_images[img_i, r, c, 0] = out_row["output_1"]
        class_output_images[img_i, r, c, 1] = out_row["output_2"]
        class_output_images[img_i, r, c, 2] = out_row["output_3"]

    # For each input_1 group, its generated images are collected.
    input_blocks.append(class_input_images)
    output_blocks.append(class_output_images)
    image_input_1.extend([class_value] * n_class_images)

print("Done")
# (total_generated_images, height, width, number_of_input_channels)
input_images = np.concatenate(input_blocks, axis=0)
# (total_generated_images, height, width, 3)
output_images = np.concatenate(output_blocks, axis=0)
# (total_generated_images,)
image_input_1 = np.array(image_input_1)

# Individual channel images, each with shape:
# (n_images, height, width, 1)
input_channel_images = [
    input_images[:, :, :, i:i+1]
    for i in range(input_images.shape[-1])
]

output_channel_images = [
    output_images[:, :, :, i:i+1]
    for i in range(output_images.shape[-1])
]

print("height:", height)
print("width:", width)
print("valid coordinate pairs:", valid_coord_mask.sum())
print("input_images shape:", input_images.shape)
print("output_images shape:", output_images.shape)
print("number of input channel images:", len(input_channel_images))
print("one input channel image shape:", input_channel_images[0].shape)
print("number of output channel images:", len(output_channel_images))
print("one output channel image shape:", output_channel_images[0].shape)
print("image_input_1 shape:", image_input_1.shape)


# Sort Hairs, longest to shortest
1) Let's pick C01. There are 68 images in this class, but shuffled up.
2) sorting each pixel in each of the 68 images, image 1 will have highest input_20 and highes output_3
   1) image 2 will have second highest input_20 and second highest output_3
   2) image 68 will have lowest input_20 and lowest highest output_3
   3) In case there is a clash with input_20 value, choose the row with higher output_3.
      1) In case there is a clash again, pick any of them
This process is repeated for all classes

In [ ]:


# -----------------------------
# Visualization
# -----------------------------

def channel_to_numeric_image(A, colname):
    flat = pd.Series(A.ravel())

    categorical_inputs = [1, 7, 8, 9, 10, 12]
    integer_inputs = [2, 3, 4, 5, 6, 11, 13, 15, 16, 17]

    categorical_cols = [f"input_{i}" for i in categorical_inputs]
    integer_cols = [f"input_{i}" for i in integer_inputs]

    if colname in categorical_cols:
        codes, _ = pd.factorize(flat, sort=True)
        codes = codes.astype(float)
        codes[codes < 0] = np.nan
        return codes.reshape(A.shape), "categorical"

    if colname in integer_cols:
        numeric = pd.to_numeric(flat, errors="coerce")
        return numeric.to_numpy().reshape(A.shape), "integer"

    numeric = pd.to_numeric(flat, errors="coerce")
    return numeric.to_numpy().reshape(A.shape), "float"


def show_generated_image(sample_idx=0):
    n_input = len(input_channel_cols)
    n_output = len(output_channel_cols)

    n_plots = n_input + n_output + 1
    n_cols = 5
    n_rows = math.ceil(n_plots / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4 * n_cols, 3.5 * n_rows),
    )

    axes = np.asarray(axes).ravel()

    plot_i = 0

    for ch, colname in enumerate(input_channel_cols):
        A = input_images[sample_idx, :, :, ch]
        img, kind = channel_to_numeric_image(A, colname)

        ax = axes[plot_i]
        im = ax.imshow(img, origin="lower", aspect="auto")
        ax.set_title(f"IN {colname} ({kind})")
        ax.set_xlabel("x index")
        ax.set_ylabel("y index")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plot_i += 1

    output_vmins = np.nanmin(output_images, axis=(0, 1, 2))
    output_vmaxs = np.nanmax(output_images, axis=(0, 1, 2))

    for ch, colname in enumerate(output_channel_cols):
        A = output_images[sample_idx, :, :, ch]

        ax = axes[plot_i]
        im = ax.imshow(
            A,
            origin="lower",
            aspect="auto",
            vmin=output_vmins[ch],
            vmax=output_vmaxs[ch],
        )

        ax.set_title(f"OUT {colname}")
        ax.set_xlabel("x index")
        ax.set_ylabel("y index")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plot_i += 1

    ax = axes[plot_i]
    im = ax.imshow(valid_coord_mask.astype(float), origin="lower", aspect="auto")
    ax.set_title("Valid coordinate-pair mask")
    ax.set_xlabel("x index")
    ax.set_ylabel("y index")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plot_i += 1

    for ax in axes[plot_i:]:
        ax.axis("off")

    fig.suptitle(
        f"Generated Image {sample_idx} | input_1 = {image_input_1[sample_idx]}",
        fontsize=16,
    )

    plt.tight_layout()
    plt.show()


# Show first generated image
show_generated_image(sample_idx=0)

In [ ]:
# -----------------------------
# GIF generation: evolution of input_20 and output_3 per input_1 class
# -----------------------------

from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image as IPythonImage

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

input_length_col = "input_20"
output_length_col = "output_3"

gif_dir = data_dir / "generated_gifs"
gif_dir.mkdir(parents=True, exist_ok=True)

fps = 1.5
dpi = 120

# ---------------------------------------------------------
# Identify channel indices
# ---------------------------------------------------------

input_20_ch = input_channel_cols.index(input_length_col)
output_3_ch = output_channel_cols.index(output_length_col)

# Ensure numeric float arrays for plotting
input_20_images = pd.to_numeric(
    pd.Series(input_images[:, :, :, input_20_ch].ravel()),
    errors="coerce"
).to_numpy().reshape(input_images[:, :, :, input_20_ch].shape)

output_3_images = output_images[:, :, :, output_3_ch].astype(float)

# All available classes, retaining construction order
class_values = pd.unique(image_input_1)

# Indices of generated images belonging to each class
class_to_indices = {
    class_value: np.flatnonzero(image_input_1 == class_value)
    for class_value in class_values
}

# ---------------------------------------------------------
# Shared colour scale
# input_20 and output_3 are both hair lengths, so the same
# colour range allows direct visual comparison.
# ---------------------------------------------------------

combined_lengths = np.concatenate([
    input_20_images[np.isfinite(input_20_images)],
    output_3_images[np.isfinite(output_3_images)]
])

length_vmin = np.nanmin(combined_lengths)
length_vmax = np.nanmax(combined_lengths)

print("Classes found:", list(class_values))
print("Shared length scale:", length_vmin, "to", length_vmax)
# ---------------------------------------------------------
# Normalised versions for coloured GIFs
# ---------------------------------------------------------

def normalise_lengths(A, vmin=length_vmin, vmax=length_vmax):
    if np.isclose(vmax, vmin):
        raise ValueError("Cannot normalise lengths because all values are identical.")

    return (A - vmin) / (vmax - vmin)


input_20_images_normalised = normalise_lengths(input_20_images)
output_3_images_normalised = normalise_lengths(output_3_images)
# ---------------------------------------------------------
# Normalised length images using one shared global scale
# ---------------------------------------------------------

def normalise_length_images(A, vmin=length_vmin, vmax=length_vmax):
    """
    Min-max normalise hair lengths to [0, 1] using the same
    global scale for input_20 and output_3.

    NaN pixels remain NaN.
    """
    if np.isclose(vmax, vmin):
        raise ValueError("Cannot normalise lengths because all values are identical.")

    return (A - vmin) / (vmax - vmin)


input_20_images_normalised = normalise_length_images(input_20_images)
output_3_images_normalised = normalise_length_images(output_3_images)

print("Normalised length range: 0 to 1")

# ---------------------------------------------------------
# Helper: draw numeric values at valid pixel coordinates
# ---------------------------------------------------------

def draw_numeric_panel(ax, A, title, decimals=0):
    """
    Display numeric values at each available coordinate.
    No heatmap colours are used.
    """
    ax.clear()

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("x index")
    ax.set_ylabel("y index")

    ax.set_xlim(-1, width)
    ax.set_ylim(-1, height)
    ax.set_aspect("equal")

    # Light grid to help track individual pixel positions
    ax.set_xticks(np.arange(-0.5, width, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, height, 1), minor=True)
    ax.grid(which="minor", linewidth=0.25)

    # Show fewer numeric axis labels so the axis is not crowded
    ax.set_xticks(np.arange(0, width, 5))
    ax.set_yticks(np.arange(0, height, 5))

    for r in range(height):
        for c in range(width):
            value = A[r, c]

            if np.isfinite(value):
                ax.text(
                    c,
                    r,
                    f"{value:.{decimals}f}",
                    ha="center",
                    va="center",
                    fontsize=5.5,
                )


# ---------------------------------------------------------
# Helper: create one numeric GIF for one class
# ---------------------------------------------------------

def create_class_gif(class_value, save_dir=gif_dir, fps=fps, dpi=dpi):
    """
    Create one GIF for a single input_1 class.

    Each animation frame corresponds to one ranked generated image:
        frame 0 = largest output_3 at each coordinate
        frame 1 = second-largest output_3 at each coordinate
        ...
    """

    sample_indices = class_to_indices[class_value]
    n_frames = len(sample_indices)

    # Larger figure because values are written inside pixels
    fig, axes = plt.subplots(1, 2, figsize=(20, 13))

    title = fig.suptitle(
        f"input_1 = {class_value} | Ranked image 0 / {n_frames - 1}",
        fontsize=15,
    )

    def update(frame_i):
        sample_idx = sample_indices[frame_i]

        draw_numeric_panel(
            axes[0],
            input_20_images[sample_idx],
            "IN input_20: initial length",
            decimals=0,
        )

        draw_numeric_panel(
            axes[1],
            output_3_images[sample_idx],
            "OUT output_3: cut length",
            decimals=0,
        )

        title.set_text(
            f"input_1 = {class_value} | "
            f"Ranked image {frame_i} / {n_frames - 1}"
        )

        return axes

    animation = FuncAnimation(
        fig,
        update,
        frames=n_frames,
        interval=1000 / fps,
        blit=False,
        repeat=True,
    )

    safe_class_name = str(class_value).replace("/", "_").replace(" ", "_")
    gif_path = save_dir / f"numeric_input_20_output_3_evolution_{safe_class_name}.gif"

    animation.save(
        gif_path,
        writer=PillowWriter(fps=fps),
        dpi=dpi,
    )

    plt.close(fig)

    return gif_path

# ---------------------------------------------------------
# Helper: create one coloured normalised GIF for one class
# ---------------------------------------------------------

def create_class_normalised_colour_gif(
    class_value,
    save_dir=gif_dir,
    fps=fps,
    dpi=dpi,
):
    """
    Create one coloured GIF for a single input_1 class.

    Values are normalised to [0, 1] using one shared scale
    across input_20 and output_3.

    No numbers are written inside the pixels.
    """

    sample_indices = class_to_indices[class_value]
    n_frames = len(sample_indices)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    first_idx = sample_indices[0]

    im_input = axes[0].imshow(
        input_20_images_normalised[first_idx],
        origin="lower",
        aspect="auto",
        vmin=0,
        vmax=1,
        cmap="viridis",
    )

    im_output = axes[1].imshow(
        output_3_images_normalised[first_idx],
        origin="lower",
        aspect="auto",
        vmin=0,
        vmax=1,
        cmap="viridis",
    )

    axes[0].set_title("IN input_20: normalised initial length")
    axes[1].set_title("OUT output_3: normalised cut length")

    for ax in axes:
        ax.set_xlabel("x index")
        ax.set_ylabel("y index")

    # One colour bar because input and output use the same [0, 1] scale
    cbar = fig.colorbar(
        im_input,
        ax=axes,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Normalised hair length")

    title = fig.suptitle(
        f"input_1 = {class_value} | Normalised colour map | "
        f"Ranked image 0 / {n_frames - 1}",
        fontsize=14,
    )

    def update(frame_i):
        sample_idx = sample_indices[frame_i]

        im_input.set_data(input_20_images_normalised[sample_idx])
        im_output.set_data(output_3_images_normalised[sample_idx])

        title.set_text(
            f"input_1 = {class_value} | Normalised colour map | "
            f"Ranked image {frame_i} / {n_frames - 1}"
        )

        return im_input, im_output, title

    animation = FuncAnimation(
        fig,
        update,
        frames=n_frames,
        interval=1000 / fps,
        blit=False,
        repeat=True,
    )

    safe_class_name = str(class_value).replace("/", "_").replace(" ", "_")
    gif_path = save_dir / f"normalised_colour_input_20_output_3_{safe_class_name}.gif"

    animation.save(
        gif_path,
        writer=PillowWriter(fps=fps),
        dpi=dpi,
    )

    plt.close(fig)

    return gif_path

# ---------------------------------------------------------
# Helper: create one normalised numeric GIF for one class
# ---------------------------------------------------------

def create_class_normalised_gif(class_value, save_dir=gif_dir, fps=fps, dpi=dpi):
    """
    Create one GIF for a single input_1 class using normalised length values.

    Both input_20 and output_3 are normalised using the same global
    length range, so their values remain directly comparable.

    Each animation frame corresponds to one ranked generated image:
        frame 0 = largest output_3 at each coordinate
        frame 1 = second-largest output_3 at each coordinate
        ...
    """

    sample_indices = class_to_indices[class_value]
    n_frames = len(sample_indices)

    fig, axes = plt.subplots(1, 2, figsize=(20, 13))

    title = fig.suptitle(
        f"input_1 = {class_value} | Normalised lengths | Ranked image 0 / {n_frames - 1}",
        fontsize=15,
    )

    def update(frame_i):
        sample_idx = sample_indices[frame_i]

        draw_numeric_panel(
            axes[0],
            input_20_images_normalised[sample_idx],
            "IN input_20: initial length normalised to [0, 1]",
            decimals=3,
        )

        draw_numeric_panel(
            axes[1],
            output_3_images_normalised[sample_idx],
            "OUT output_3: cut length normalised to [0, 1]",
            decimals=3,
        )

        title.set_text(
            f"input_1 = {class_value} | Normalised lengths | "
            f"Ranked image {frame_i} / {n_frames - 1}"
        )

        return axes

    animation = FuncAnimation(
        fig,
        update,
        frames=n_frames,
        interval=1000 / fps,
        blit=False,
        repeat=True,
    )

    safe_class_name = str(class_value).replace("/", "_").replace(" ", "_")
    gif_path = save_dir / f"normalised_numeric_input_20_output_3_evolution_{safe_class_name}.gif"

    animation.save(
        gif_path,
        writer=PillowWriter(fps=fps),
        dpi=dpi,
    )

    plt.close(fig)

    return gif_path


# ---------------------------------------------------------
# Generate one numeric GIF for every class
# ---------------------------------------------------------
individual_gif_paths = []
normalised_colour_gif_paths = []

for class_value in class_values:
    # Existing number-only GIF
    numeric_gif_path = create_class_gif(class_value)
    individual_gif_paths.append(numeric_gif_path)
    print(f"Saved numeric GIF: {numeric_gif_path}")

    # New coloured GIF with values normalised to [0, 1]
    colour_gif_path = create_class_normalised_colour_gif(class_value)
    normalised_colour_gif_paths.append(colour_gif_path)
    print(f"Saved normalised colour GIF: {colour_gif_path}")

print(f"\nCreated {len(individual_gif_paths)} numeric GIFs.")
print(f"Created {len(normalised_colour_gif_paths)} normalised colour GIFs.")